## Data analysis

In [24]:
import sys
import os

# Add the parent directory (simcode) to sys.path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

In [25]:
import pandas as pd
import numpy as np
import json
from ecv_rank import *
from src.solvers.weighted_network import ASE
from src.methods import * 
import matplotlib.pyplot as plt
import requests

### County migration dataclean

In [ ]:
county_migration = pd.read_excel(
    'county-to-county-2016-2020-ins-outs-nets-gross.xlsx',
    sheet_name=None
)

# --------------------------------------------------
# Clean individual sheets
# --------------------------------------------------

def clean_county_name(s):
    s = s.str.lower()

    for text in ['county', '.', 'borough', 'census area', 'parish', 'city', ' ']:
        s = s.str.replace(text, '', regex=False)

    return s


def clean_sheet(df):

    # First row contains column names
    cols = list(df.iloc[0])

    # Replace unnamed MOE columns
    for i in range(len(cols)):
        if pd.isna(cols[i]):
            cols[i] = str(cols[i - 1]) + '_moe'

    # Remove first two rows and last six rows
    df = df.iloc[2:-6].copy()
    df.columns = cols

    # Remove rows without Geography B county FIPS
    df = df[df['FIPS County Code of Geography B'].notna()].copy()

    # FIPS county codes
    df['FIPS County Code of Geography A'] = (
        df['FIPS County Code of Geography A'].astype(int)
    )

    df['FIPS County Code of Geography B'] = (
        df['FIPS County Code of Geography B'].astype(int)
    )

    # State names
    df['State Name of Geography A'] = (
        df['State Name of Geography A'].str.lower()
    )

    df['State/U.S. Island Area/Foreign Region of Geography B'] = (
        df['State/U.S. Island Area/Foreign Region of Geography B'].str.lower()
    )

    # County names
    df['County Name of Geography A'] = clean_county_name(
        df['County Name of Geography A']
    )

    df['County Name of Geography B'] = clean_county_name(
        df['County Name of Geography B']
    )

    return df


# --------------------------------------------------
# Clean all sheets
# --------------------------------------------------

county_migration_clean = {
    sheet_name.lower(): clean_sheet(df)
    for sheet_name, df in county_migration.items()
}

# Remove Puerto Rico sheet
county_migration_clean.pop('puerto rico')

# Remove flows where Geography B is Puerto Rico
county_migration_clean = {
    key: df[
        df['State/U.S. Island Area/Foreign Region of Geography B'] != 'puerto rico'
    ]
    for key, df in county_migration_clean.items()
}


# --------------------------------------------------
# Combine sheets
# --------------------------------------------------

county_migration_combined = pd.concat(
    county_migration_clean.values(),
    ignore_index=True
)


# --------------------------------------------------
# Rename columns
# --------------------------------------------------

county_migration_combined = county_migration_combined.rename(columns={
    'State Code of Geography A': 'state_code',
    'FIPS County Code of Geography A': 'county_code',

    'State/U.S. Island Area/Foreign Region Code of Geography B': 'state_code_B',
    'FIPS County Code of Geography B': 'county_code_B',

    'State Name of Geography A': 'state_name',
    'County Name of Geography A': 'county_name',

    'State/U.S. Island Area/Foreign Region of Geography B': 'state_name_B',
    'County Name of Geography B': 'county_name_B',

    'Gross Migration between Geography A and Geography B1': 'gross_migration',
    'Net Migration between Geography A and Geography B1': 'net_migration'
})


# --------------------------------------------------
# Format FIPS codes
# --------------------------------------------------

county_migration_combined['county_code'] = (
    county_migration_combined['county_code']
    .astype(str)
    .str.zfill(3)
)

county_migration_combined['county_code_B'] = (
    county_migration_combined['county_code_B']
    .astype(str)
    .str.zfill(3)
)


# --------------------------------------------------
# Remove Alaska and Kalawao
# --------------------------------------------------

county_migration_combined = county_migration_combined[
    (county_migration_combined['state_name'] != 'alaska') &
    (county_migration_combined['state_name_B'] != 'alaska') &
    (county_migration_combined['state_name'] != 'hawaii') & 
    (county_migration_combined['state_name_B'] != 'hawaii')
].copy()

In [ ]:
# GDP data combines some counties together so we need to combine the aswell

merge_groups = {
    '51540': ['51003', '51540'],            # Albemarle + Charlottesville
    '51005': ['51005', '51580'],            # Alleghany + Covington
    '51790': ['51015', '51790', '51820'],   # Augusta + Staunton + Waynesboro
    '51031': ['51031', '51680'],            # Campbell + Lynchburg
    '51035': ['51035', '51640'],            # Carroll + Galax
    '51570': ['51053', '51570', '51730'],   # Dinwiddie + Colonial Heights + Petersburg
    '51610': ['51059', '51600', '51610'],   # Fairfax + Fairfax City + Falls Church
    '51069': ['51069', '51840'],            # Frederick + Winchester
    '51081': ['51081', '51595'],            # Greensville + Emporia
    '51690': ['51089', '51690'],            # Henry + Martinsville
    '51095': ['51095', '51830'],            # James City + Williamsburg
    '51750': ['51121', '51750'],            # Montgomery + Radford
    '51143': ['51143', '51590'],            # Pittsylvania + Danville
    '51670': ['51149', '51670'],            # Prince George + Hopewell
    '51685': ['51153', '51683', '51685'],   # Prince William + Manassas + Manassas Park
    '51775': ['51161', '51775'],            # Roanoke + Salem
    '51163': ['51163', '51530', '51678'],   # Rockbridge + Buena Vista + Lexington
    '51165': ['51165', '51660'],            # Rockingham + Harrisonburg
    '51175': ['51175', '51620'],            # Southampton + Franklin
    '51177': ['51177', '51630'],            # Spotsylvania + Fredericksburg
    '51191': ['51191', '51520'],            # Washington + Bristol
    '51195': ['51195', '51720'],            # Wise + Norton
    '51199': ['51199', '51735'],            # York + Poquoson
}

recast_dict = {
    '51901': '51540', 
    '51903': '51005',
    '51907': '51790',
    '51911': '51031',
    '51913': '51035',
    '51918': '51570',
    '51919': '51610',
    '51921': '51069',
    '51923': '51081',
    '51929': '51690',
    '51931': '51095',
    '51933': '51750',
    '51939': '51143',
    '51941': '51670',
    '51942': '51685',
    '51944': '51775',
    '51945': '51163',
    '51947': '51165',
    '51949': '51175',
    '51951': '51177',
    '51953': '51191',
    '51955': '51195',
    '51958': '51199'
}


sum_cols = [
    'Flow from Geography B to Geography A',
    'Flow from Geography B to Geography A_moe',
    'Counterflow from Geography A to Geography B1',
    'Counterflow from Geography A to Geography B1_moe',
    'Net Migration from Geography B to Geography A1',
    'Net Migration from Geography B to Geography A1_moe', 'gross_migration',
    'Gross Migration between Geography A and Geography B1_moe'
]

group_cols = [
    'state_name',
    'state_code_B',
    'county_code_B',
    'state_name_B',
    'county_name_B'
]

def merge_county_groups(df, merge_groups, sum_cols, group_cols):
    df = df.copy()

    # Keep FIPS codes as zero-padded strings
    df['state_code'] = df['state_code'].astype(str).str.zfill(3)
    df['county_code'] = df['county_code'].astype(str).str.zfill(3)

    # Map each original FIPS to its merged FIPS
    fips_map = {
        old_fips: new_fips
        for new_fips, old_fips_list in merge_groups.items()
        for old_fips in old_fips_list
    }

    # Include old GDP FIPS codes in the mapping, if present
    fips_map.update(recast_dict)

    # Match on full 5-digit Virginia FIPS
    full_fips = (
        df['state_code'].str[-2:] +
        df['county_code']
    )

    mask = full_fips.isin(fips_map)

    merged = df.loc[mask].copy()

    # Assign canonical county code
    merged['county_code'] = (
        full_fips.loc[mask]
        .map(fips_map)
        .str[-3:]
    )

    # Remove county_name because it would prevent merging
    merge_keys = group_cols + ['state_code', 'county_code']

    merged = (
        merged.groupby(
            merge_keys,
            as_index=False,
            dropna=False
        )[sum_cols]
        .sum(min_count=1)
    )

    # Assign a consistent county name for each merged FIPS
    merged['county_name'] = merged['county_code']

    # Preserve all rows outside the merge groups
    result = pd.concat(
        [df.loc[~mask], merged],
        ignore_index=True
    )

    return result


county_migration_merged = merge_county_groups(
    county_migration_combined,
    merge_groups=merge_groups,
    sum_cols=sum_cols,
    group_cols=group_cols
)

In [74]:
county_migration_merged.to_csv('data_cleaned/county_migration_cleaned.csv', index=False, )

### voting data

In [32]:
# ============================================================
# Load data
# ============================================================

county_voting = pd.read_csv(
    'dataverse_files/countypres_2000-2024.csv'
)


# ============================================================
# Basic filtering
# ============================================================

county_voting_clean = (
    county_voting.loc[
        (county_voting['state'] != 'ALASKA') &
        (county_voting['state'] != 'HAWAII') &
        (county_voting['year'].isin([2012, 2016, 2020])) &
        (county_voting['totalvotes'] != 0) &
        (county_voting['county_fips'].notna())
    ]
    .copy()
)


# ============================================================
# Clean county names
# ============================================================

def clean_county_name(name):
    name = name.lower()

    for text in [
        'county',
        'borough',
        'census area',
        'parish',
        'city',
        '.',
        ' '
    ]:
        name = name.replace(text, '')

    name = name.replace('saint', 'st')

    # Special cases
    if name == 'donaana':
        name = 'doñaana'

    if name == 'kansas':
        name = 'jackson'

    return name


county_voting_clean['county_name'] = (
    county_voting_clean['county_name']
    .apply(clean_county_name)
)

county_voting_clean['state'] = (
    county_voting_clean['state']
    .str.lower()
)

county_voting_clean = county_voting_clean.rename(
    columns={'state': 'state_name'}
)


# ============================================================
# Create state and county FIPS codes
# ============================================================

fips = (
    county_voting_clean['county_fips']
    .astype(str)
    .str.replace('.0', '', regex=False)
)

county_voting_clean['county_code'] = (
    fips.str[-3:]
)

county_voting_clean['state_code'] = (
    fips.str[:-3]
    .str.zfill(3)
)


# ============================================================
# South Dakota special case
# Shannon County -> Oglala Lakota County
# ============================================================

mask = (
    (county_voting_clean['county_name'] == 'shannon') &
    (county_voting_clean['state_name'] == 'south dakota')
)

county_voting_clean.loc[mask, 'county_code'] = '102'
county_voting_clean.loc[mask, 'county_name'] = 'oglalalakota'


# ============================================================
# Helper for combining duplicate geographic observations
# ============================================================

def combine_counties(
    df,
    source_mask,
    target_mask,
    group_cols,
    drop_mask=None
):
    """
    Aggregate all rows identified by source_mask and assign the
    aggregated vote totals to the rows identified by target_mask.
    Optionally remove unwanted duplicate geography afterward.
    """

    totals = (
        df.loc[source_mask]
        .groupby(group_cols, as_index=False)
        .agg({
            'candidatevotes': 'sum',
            'totalvotes': 'sum'
        })
    )

    df.loc[target_mask, ['totalvotes', 'candidatevotes']] = (
        df.loc[target_mask, group_cols]
        .merge(
            totals,
            on=group_cols,
            how='left'
        )[['totalvotes', 'candidatevotes']]
        .to_numpy()
    )

    if drop_mask is not None:
        df = df.loc[~drop_mask].copy()

    return df


# ============================================================
# Bedford, Virginia
# Combine Bedford County + Bedford city
# Keep county code 019 and remove code 515
# ============================================================

source_mask = (
    (county_voting_clean['county_name'] == 'bedford') &
    (county_voting_clean['state_name'] == 'virginia')
)

target_mask = (
    source_mask &
    (county_voting_clean['county_code'] == '019')
)

drop_mask = (
    source_mask &
    (county_voting_clean['county_code'] == '515')
)

county_voting_clean = combine_counties(
    df=county_voting_clean,
    source_mask=source_mask,
    target_mask=target_mask,
    group_cols=['year', 'candidate', 'party', 'mode'],
    drop_mask=drop_mask
)


# ============================================================
# Jackson, Missouri
# Combine matching Jackson observations
# ============================================================

source_mask = (
    (county_voting_clean['county_name'] == 'jackson') &
    (county_voting_clean['state_name'] == 'missouri')
)

target_mask = (
    (county_voting_clean['county_code'] == '095') &
    (county_voting_clean['state_code'] == '029')
)

county_voting_clean = combine_counties(
    df=county_voting_clean,
    source_mask=source_mask,
    target_mask=target_mask,
    group_cols=['year', 'candidate', 'party']
)


# ============================================================
# Remove unwanted observation
# ============================================================

county_voting_clean = county_voting_clean.loc[
    ~(
        (county_voting_clean['county_code'] == '000') &
        (county_voting_clean['state_code'] == '2938')
    )
].copy()

In [33]:
county_voting_clean.to_csv('data_cleaned/county_voting_cleaned.csv', index=False)

In [34]:
votes_change = (
    county_voting_clean[
        county_voting_clean['party'].isin(['DEMOCRAT', 'REPUBLICAN'])
    ]
    .groupby(
        ['state_code', 'county_code', 'party', 'year'],
        as_index=False
    )['candidatevotes']
    .sum()
)

# Put party/year combinations into columns
votes_change = (
    votes_change
    .pivot(
        index=['state_code', 'county_code'],
        columns=['party', 'year'],
        values='candidatevotes'
    )
)

# Compute changes
votes_change['dem_change_2012_2016'] = (
    votes_change[('DEMOCRAT', 2016)]
    - votes_change[('DEMOCRAT', 2012)]
)

votes_change['dem_change_2016_2020'] = (
    votes_change[('DEMOCRAT', 2020)]
    - votes_change[('DEMOCRAT', 2016)]
)

votes_change['rep_change_2012_2016'] = (
    votes_change[('REPUBLICAN', 2016)]
    - votes_change[('REPUBLICAN', 2012)]
)

votes_change['rep_change_2016_2020'] = (
    votes_change[('REPUBLICAN', 2020)]
    - votes_change[('REPUBLICAN', 2016)]
)

# Keep only the change variables
votes_change = (
    votes_change[
        [
            'dem_change_2012_2016',
            'dem_change_2016_2020',
            'rep_change_2012_2016',
            'rep_change_2016_2020'
        ]
    ]
    .reset_index()
)

In [35]:
votes_change.to_csv('data_cleaned/votes_change.csv', index=False)

### GDP per capita 

In [9]:
import requests
import pandas as pd


def get_county_gdp_per_capita(
    api_key,
    start_year=2001,
    end_year=2024
):
    """
    Download county-level real GDP and population from BEA
    and calculate real GDP per capita.

    Returns one row per county-year.

    real_gdp_per_capita is in chained 2017 dollars per person.
    """

    base_url = "https://apps.bea.gov/api/data/"

    years = ",".join(
        str(year) for year in range(start_year, end_year + 1)
    )

    def get_bea_data(table, line_code):

        params = {
            "UserID": api_key,
            "method": "GetData",
            "datasetname": "Regional",
            "TableName": table,
            "LineCode": line_code,
            "GeoFips": "COUNTY",
            "Year": years,
            "ResultFormat": "JSON"
        }

        response = requests.get(
            base_url,
            params=params,
            timeout=60
        )

        response.raise_for_status()

        result = response.json()["BEAAPI"]["Results"]

        # Catch BEA API errors
        if "Error" in result:
            raise ValueError(result["Error"])

        return pd.DataFrame(result["Data"])


    # --------------------------------------------------
    # 1. Real GDP
    # CAGDP1 line 1:
    # Real GDP, thousands of chained 2017 dollars
    # --------------------------------------------------

    gdp = get_bea_data(
        table="CAGDP1",
        line_code=1
    )

    # --------------------------------------------------
    # 2. Population
    # CAINC1 line 2:
    # Population
    # --------------------------------------------------

    population = get_bea_data(
        table="CAINC1",
        line_code=2
    )


    # --------------------------------------------------
    # Clean FIPS
    # --------------------------------------------------

    gdp["county_fips"] = (
        gdp["GeoFips"]
        .astype(str)
        .str.strip()
        .str.zfill(5)
    )

    population["county_fips"] = (
        population["GeoFips"]
        .astype(str)
        .str.strip()
        .str.zfill(5)
    )


    # --------------------------------------------------
    # Clean years
    # --------------------------------------------------

    gdp["year"] = gdp["TimePeriod"].astype(int)

    population["year"] = (
        population["TimePeriod"]
        .astype(int)
    )


    # --------------------------------------------------
    # Convert BEA values from strings to numeric
    #
    # BEA returns values such as:
    # "1,234,567"
    # --------------------------------------------------

    gdp["real_gdp_thousands"] = pd.to_numeric(
        gdp["DataValue"].str.replace(",", "", regex=False),
        errors="coerce"
    )

    population["population"] = pd.to_numeric(
        population["DataValue"].str.replace(",", "", regex=False),
        errors="coerce"
    )


    # --------------------------------------------------
    # Keep variables we need
    # --------------------------------------------------

    gdp = gdp[
        [
            "county_fips",
            "GeoName",
            "year",
            "real_gdp_thousands"
        ]
    ].rename(
        columns={"GeoName": "county_name"}
    )

    population = population[
        [
            "county_fips",
            "year",
            "population"
        ]
    ]


    # --------------------------------------------------
    # Merge GDP + population
    # --------------------------------------------------

    df = gdp.merge(
        population,
        on=["county_fips", "year"],
        how="left",
        validate="1:1"
    )


    # --------------------------------------------------
    # Calculate real GDP per capita
    #
    # GDP is reported in thousands of dollars,
    # hence * 1000
    # --------------------------------------------------

    df["real_gdp"] = (
        df["real_gdp_thousands"] * 1000
    )

    df["real_gdp_per_capita"] = (
        df["real_gdp"] / df["population"]
    )


    # --------------------------------------------------
    # Extract state and county FIPS components
    #
    # 06037:
    #   state_code  = 06
    #   county_code = 037
    # --------------------------------------------------

    df["state_code"] = df["county_fips"].str[:2]

    df["county_code"] = df["county_fips"].str[-3:]


    return df.sort_values(
        ["county_fips", "year"]
    ).reset_index(drop=True)

In [20]:
API_KEY = "AC9D5FE2-DFA3-4463-B27D-0B5E8702E8FF"

county_gdp = get_county_gdp_per_capita(
    API_KEY,
    start_year=2012,
    end_year=2020
)

In [21]:
county_gdp.drop(
    county_gdp[county_gdp['state_code'] == '02'].index,
    inplace=True
)

county_gdp.drop(
    county_gdp[county_gdp['state_code'] == '72'].index,
    inplace=True
)

county_gdp.drop(
    county_gdp[county_gdp['state_code'] == '15'].index,
    inplace=True
)


In [39]:
county_gdp[(county_gdp['state_code'] == '51') & (county_gdp['county_code'] == '3')]

,county_fips,county_name,year,real_gdp_thousands,population,real_gdp,real_gdp_per_capita,state_code,county_code


In [ ]:
# recast new county code

recast_dict = {
    '51901': '51540', 
    '51903': '51005',
    '51907': '51790',
    '51911': '51031',
    '51913': '51035',
    '51918': '51570',
    '51919': '51610',
    '51921': '51069',
    '51923': '51081',
    '51929': '51690',
    '51931': '51095',
    '51933': '51750',
    '51939': '51143',
    '51941': '51670',
    '51942': '51685',
    '51944': '51775',
    '51945': '51163',
    '51947': '51165',
    '51949': '51175',
    '51951': '51177',
    '51953': '51191',
    '51955': '51195',
    '51958': '51199'
}

for old_fips, new_fips in recast_dict.items():
    county_gdp.loc[county_gdp['county_fips'] == old_fips, 'county_fips'] = new_fips

In [ ]:
for old_fips, new_fips in recast_dict.items():
    print(county_gdp[county_gdp['county_fips'] == new_fips].iloc[0]['county_name'])



Albemarle + Charlottesville, VA*
Alleghany + Covington, VA*
Augusta, Staunton + Waynesboro, VA*
Campbell + Lynchburg, VA*
Carroll + Galax, VA*
Dinwiddie, Colonial Heights + Petersburg, VA*
Fairfax, Fairfax City + Falls Church, VA*
Frederick + Winchester, VA*
Greensville + Emporia, VA*
Henry + Martinsville, VA*
James City + Williamsburg, VA*
Montgomery + Radford, VA*
Pittsylvania + Danville, VA*
Prince George + Hopewell, VA*
Prince William, Manassas + Manassas Park, VA*
Roanoke + Salem, VA*
Rockbridge, Buena Vista + Lexington, VA*
Rockingham + Harrisonburg, VA*
Southampton + Franklin, VA*
Spotsylvania + Fredericksburg, VA*
Washington + Bristol, VA*
Wise + Norton, VA*
York + Poquoson, VA*


,county_fips,county_name,year,real_gdp_thousands,population,real_gdp,real_gdp_per_capita,state_code,county_code


In [13]:
county_gdp.to_csv('data_cleaned/county_gdp_per_capita_2012_2020.csv', index=False)

In [14]:
gdp_change = (
    county_gdp[county_gdp["year"].isin([2012, 2016])]
    .pivot(
        index="county_fips",
        columns="year",
        values="real_gdp_per_capita"
    )
)

gdp_change["gdp_pc_change_2012_2016"] = (
    gdp_change[2016] - gdp_change[2012]
)

gdp_change = gdp_change.reset_index()

In [15]:
gdp_change.to_csv('data_cleaned/gdp_change_2012_2016.csv', index=False)

## Dataset build

In [16]:
# ============================================================
# Create county codes for Geography A and Geography B
# ============================================================

county_migration_combined['code_A'] = (
    county_migration_combined['state_code'].astype(str)
    + '_'
    + county_migration_combined['county_code'].astype(str).str.zfill(3)
)

county_migration_combined['code_B'] = (
    county_migration_combined['state_code_B'].astype(str)
    + '_'
    + county_migration_combined['county_code_B'].astype(str).str.zfill(3)
)


# ============================================================
# Create county code <-> county name mappings
# ============================================================

df_names = (
    county_migration_combined
    .groupby('code_A')[['state_name', 'county_name']]
    .first()
    .rename(columns={
        'state_name': 'State Name',
        'county_name': 'County Name'
    })
)


codetoname = {
    code: row['State Name'] + '_' + row['County Name'].replace(' ', '_')
    for code, row in df_names.iterrows()
}

nametocode = {
    name: code
    for code, name in codetoname.items()
}


# ============================================================
# Create index mappings
# ============================================================

nametoidx = {
    name: i
    for i, name in enumerate(nametocode.keys())
}

idxtoname = {
    i: name
    for name, i in nametoidx.items()
}


codetoidx = {
    code: i
    for i, code in enumerate(codetoname.keys())
}

idxtocode = {
    i: code
    for code, i in codetoidx.items()
}


# ============================================================
# Create adjacency matrix
# ============================================================

n = county_migration_combined.groupby('state_name')['county_code'].nunique().sum()

adj_matrix = np.zeros((n + 1, n + 1))

interactions = 0


for _, row in county_migration_combined.iterrows():

    i = codetoidx[row['code_A']]
    j = codetoidx[row['code_B']]

    adj_matrix[j, i] = row['gross_migration']

    interactions += 1


# ============================================================
# Total migration by destination
# ============================================================

d = adj_matrix.sum(axis=1)

eps = 1e-10

In [17]:
codetoname = {}
df_names = pd.DataFrame()

for key in county_migration_clean.keys():
    county_migration_clean[key]['code_A'] = (
        county_migration_clean[key]['State Code of Geography A'].astype(str)
        + '_'
        + county_migration_clean[key]['FIPS County Code of Geography A'].astype(str).str.zfill(3)
    )

    county_migration_clean[key]['code_B'] = (
        county_migration_clean[key]['State/U.S. Island Area/Foreign Region Code of Geography B'].astype(str)
        + '_'
        + county_migration_clean[key]['FIPS County Code of Geography B'].astype(str).str.zfill(3)
    )
    tt = county_migration_clean[key].groupby('code_A')[['State Name of Geography A', 'County Name of Geography A']].first()
    for index, row in tt.iterrows():
        df_names.loc[index, 'State Name'] = row['State Name of Geography A']
        df_names.loc[index, 'County Name'] = row['County Name of Geography A']
        codetoname[index] = row['State Name of Geography A'] + '_' + row['County Name of Geography A'].replace(' ', '_')

nametocode = {v: k for k, v in codetoname.items()}

nametoidx = {k: i for i, k in enumerate(nametocode.keys())}
idxtoname = {v: k for k, v in nametoidx.items()}

codetoidx = {k: i for i, k in enumerate(codetoname.keys())}
idxtocode = {v: k for k, v in codetoidx.items()}

In [18]:

n, d = 0, 0

for key in county_migration_clean.keys():
    n += county_migration_clean[key]['FIPS County Code of Geography A'].nunique()
    d += county_migration_clean[key]['FIPS County Code of Geography B'].nunique()
adj_matrix = np.zeros((n+1, n+1))
interactions = 0

for key in county_migration_clean.keys():
    for index, row in county_migration_clean[key].iterrows():
        i = codetoidx[row['code_A']]
        j = codetoidx[row['code_B']]
        adj_matrix[j, i] = row['Gross Migration between Geography A and Geography B1']
        interactions += 1
d = adj_matrix.sum(axis=1)
eps = 1e-10

# dinv = 1 / np.sqrt(d + eps)

# L = dinv[:, None] * adj_matrix * dinv[None, :]
# result = ecv_rank(
#     L,
#     max_k=100,
#     B=10,
#     holdout_p=0.1,
#     weighted=True,
#     mode="undirected",
#     random_state=2,
# )

# print("SSE selected rank:", result.sse_rank)
# print("AUC selected rank:", result.auc_rank)

# plt.plot(result.sse)
# from scipy.sparse.linalg import eigsh

# evals, evecs = eigsh(L, k=result.sse_rank, which="LM")
# plt.plot(np.sort(np.abs(evals))[::-1])